In [ ]:
!uv add langchain-deepseek==1.0.1

Resolved 158 packages in 1.62s
 Downloaded openai
Prepared 6 packages in 998ms
Installed 9 packages in 257ms
 + distro==1.9.0
 + jiter==0.15.0
 + langchain-deepseek==1.0.1
 + langchain-openai==1.2.2
 + openai==2.38.0
 + regex==2026.5.9
 + sniffio==1.3.1
 + tiktoken==0.13.0
 + tqdm==4.67.3


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
from langchain_community.chat_models import ChatTongyi

# 通义模型
tongyi_key =  os.environ.get('QWEN_KEY')
os.environ["DASHSCOPE_API_KEY"] = tongyi_key

llm = ChatTongyi()

from langchain_deepseek.chat_models import ChatDeepSeek


deepseek_key=os.environ.get('DEEPSEEK_KEY')
os.environ["DEEPSEEK_API_KEY"]=deepseek_key

ds_llm = ChatDeepSeek(model='deepseek-chat')

### 综合Agent：Tools + Memory + Graph

In [ ]:
import json

from langchain_core.tools import tool
from langchain_core.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langchain_core.messages import HumanMessage

# ===== 1. 工具定义 =====

store = InMemoryStore()

@tool
def save_note(user_id: str, title: str, content: str) -> str:
    """保存用户笔记。user_id是用户ID，title是笔记标题，content是笔记内容。"""
    namespace = ("notes", user_id)
    notes = []
    existing = store.get(namespace, "all_notes")
    if existing:
        notes = existing.value
    notes.append({"title": title, "content": content})
    store.put(namespace, "all_notes", notes)
    return f"笔记'{title}'已保存"

@tool
def get_notes(user_id: str) -> str:
    """查询用户所有笔记。user_id是用户ID。"""
    namespace = ("notes", user_id)
    existing = store.get(namespace, "all_notes")
    if not existing or not existing.value:
        return "暂无笔记"
    return json.dumps(existing.value, ensure_ascii=False)

@tool
def save_user_profile(user_id: str, key: str, value: str) -> str:
    """保存用户个人信息。key是信息类别（name/age/city/hobby），value是内容。"""
    namespace = ("users", user_id)
    existing = store.get(namespace, "profile")
    profile = existing.value if existing else {}
    profile[key] = value
    store.put(namespace, "profile", profile)
    return f"已保存：{key} = {value}"

@tool
def get_user_profile(user_id: str) -> str:
    """查询用户个人信息。"""
    namespace = ("users", user_id)
    existing = store.get(namespace, "profile")
    if not existing:
        return "暂无用户信息"
    return json.dumps(existing.value, ensure_ascii=False)

@tool
def calculator(expression: str) -> str:
    """计算数学表达式。"""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"计算错误：{e}"

# ===== 2. 系统提示词 =====

SYSTEM_PROMPT = """你是一个智能个人助手，具备以下能力：
1. 记住用户的个人信息（姓名、年龄、城市、爱好等）
2. 帮用户记录和查询笔记
3. 进行数学计算

当用户提到自己的个人信息时，请主动调用 save_user_profile 保存。
用户ID固定为 "user-001"。
回答要简洁友好。"""

# ===== 3. 创建Agent（Tools + Memory + Graph 一步到位） =====
app = create_agent(
    ds_llm,
    [save_note, get_notes, save_user_profile, get_user_profile, calculator],
    checkpointer=MemorySaver(), # 短期记忆
    system_prompt=SYSTEM_PROMPT
)

config = {"configurable": {"thread_id": "assistant-001"}}

# ===== 4. 多轮对话 =====

conversations = [
    "你好，我叫张三，今年25岁，住在北京",
    "帮我记一条笔记：明天下午3点开会",
    "帮我算一下，如果我每天存50元，一年能存多少？",
    "我之前记了什么笔记？",
    "你还记得我的个人信息吗？",
]

for msg in conversations:
    result = app.invoke(
        {"messages": [HumanMessage(msg)]},
        config
    )
    print(f"用户：{msg}")
    print(f"AI：{result['messages'][-1].content}")
    print()

用户：你好，我叫张三，今年25岁，住在北京
AI：已经记住啦！
- 👤 **姓名**：张三
- 🎂 **年龄**：25岁
- 📍 **城市**：北京

有什么我可以帮你的吗？比如记录笔记、查询信息或者算个数学题都可以哦~

用户：帮我记一条笔记：明天下午3点开会
AI：已经记好啦 ✅ 笔记内容：**明天下午3点开会**

还有其他需要帮忙的吗？比如想查看之前的笔记，或者算个啥的~ 😊

用户：帮我算一下，如果我每天存50元，一年能存多少？
AI：按一年365天算，每天存 **50元**，一年下来可以存 **18,250元** 💰

坚持存钱是个好习惯，加油哦，张三！💪 还有其他需要帮忙的吗？😊

用户：我之前记了什么笔记？
AI：目前你只有一条笔记 📝：

📌 **开会提醒** — 明天下午3点开会

别忘了明天的会议哦！需要再记点什么吗？😊

用户：你还记得我的个人信息吗？
AI：当然记得啦！你在我这里的档案是：

- 👤 **姓名**：张三
- 🎂 **年龄**：25岁
- 📍 **城市**：北京

还有什么需要帮忙的吗？😊



### 多Agent协作

In [3]:
from typing import Annotated, Sequence, TypedDict
from operator import add

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, END, START
from langchain_core.agents import create_agent


# ===== 1. 定义专业Agent =====
# 技术支持Agent
@tool
def check_system_status(service_name: str) -> str:
    """检查系统服务状态，输入服务名称。"""
    status = {
        "数据库": "正常运行，响应时间50ms",
        "API网关": "正常运行，QPS 1200",
        "缓存服务": "告警：内存使用率85%"
    }
    return status.get(service_name, f"{service_name}：未找到该服务")

@tool
def restart_service(service_name: str) -> str:
    """重启指定服务。"""
    return f"服务 {service_name} 已重启，预计30秒恢复"

tech_tools = [check_system_status, restart_service]
tech_agent = create_agent(
    ds_llm, tech_tools,
    system_prompt="你是技术支持专家，负责检查系统状态和处理技术问题。回答要专业简洁。"
)


# 客服Agent
@tool
def query_order(order_id: str) -> str:
    """查询订单信息，输入订单号。"""
    orders = {
        "ORD001": {"status": "已发货", "amount": "299元", "delivery": "预计明天到达"},
        "ORD002": {"status": "处理中", "amount": "1599元", "delivery": "预计3天后发货"},
    }
    order = orders.get(order_id)
    if order:
        return f"订单{order_id}：状态-{order['status']}，金额-{order['amount']}，{order['delivery']}"
    return f"未找到订单 {order_id}"

@tool
def process_refund(order_id: str, reason: str) -> str:
    """处理退款申请，输入订单号和退款原因。"""
    return f"订单{order_id}的退款申请已提交（原因：{reason}），预计1-3个工作日到账"

cs_tools = [query_order, process_refund]
cs_agent = create_agent(
    ds_llm, cs_tools,
    system_prompt="你是客服专员，负责处理订单查询和退款申请。态度友好，耐心解答。"
)

# ===== 2. 定义 Supervisor =====

class SupervisorState(TypedDict):
    messages: Annotated[Sequence, add]
    next_agent: str

def supervisor_node(state: SupervisorState):
    """Supervisor：分析用户问题，决定交给哪个Agent处理"""
    last_message = state["messages"][-1]
    user_text = last_message.content if hasattr(last_message, "content") else str(last_message)

    # 使用LLM判断应该交给哪个Agent
    prompt = f"""根据用户的问题，判断应该交给哪个团队处理：

选项：
- tech：技术支持团队（系统故障、服务状态、技术问题）
- cs：客服团队（订单查询、退款、售后服务）
- direct：直接回答（问候、简单问答）

用户问题：{user_text}

只回复选项名称（tech/cs/direct），不要其他内容。"""

    response = ds_llm.invoke(prompt)
    choice = response.content.strip().lower()

    if choice not in ("tech", "cs"):
        choice = "direct"

    return {"next_agent": choice}


def tech_node(state: SupervisorState):
    """调用技术支持Agent"""
    user_msg = state["messages"][-1]
    result = tech_agent.invoke({"messages": [user_msg]})
    return {"messages": [result["messages"][-1]]}

def cs_node(state: SupervisorState):
    """调用客服Agent"""
    user_msg = state["messages"][-1]
    result = cs_agent.invoke({"messages": [user_msg]})
    return {"messages": [result["messages"][-1]]}

def direct_node(state: SupervisorState):
    """直接回答"""
    user_msg = state["messages"][-1]
    response = ds_llm.invoke(f"你是一个友好的客服中心接待员，请回答：{user_msg.content if hasattr(user_msg, 'content') else user_msg}")
    return {"messages": [AIMessage(response.content)]}

def route_to_agent(state: SupervisorState) -> str:
    return state["next_agent"]

# ===== 3. 构建Supervisor Graph =====

graph = StateGraph(SupervisorState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("tech", tech_node)
graph.add_node("cs", cs_node)
graph.add_node("direct", direct_node)

graph.add_edge(START, "supervisor")
graph.add_conditional_edges("supervisor", route_to_agent, {
    "tech": "tech",
    "cs": "cs",
    "direct": "direct"
})
graph.add_edge("tech", END)
graph.add_edge("cs", END)
graph.add_edge("direct", END)

app = graph.compile()

# ===== 4. 测试多Agent =====

questions = [
    "你好，请问你们的工作时间是什么？",
    "帮我查一下ORD001订单到哪了",
    "缓存服务有告警，帮我看看",
    "我想退款ORD002，商品有质量问题",
]

for question in questions:
    result = app.invoke({"messages": [HumanMessage(question)]})
    print(f"用户：{question}")
    print(f"路由：{result['next_agent']}")
    print(f"回复：{result['messages'][-1].content}")
    print('='*20)

用户：你好，请问你们的工作时间是什么？
路由：direct
回复：您好！很高兴为您服务！😊

我们的工作时间是 **每天 9:00 - 21:00**（包括周末和节假日），这段时间内都会有客服人员在线为您提供帮助。

如果您在这个时间之外留言，我们会在下一个工作日第一时间回复您。

请问还有什么可以帮您的吗？
用户：帮我查一下ORD001订单到哪了
路由：cs
回复：您好！为您查询到订单 **ORD001** 的最新状态如下：

- **订单状态**：✅ 已发货
- **订单金额**：299 元
- **预计送达**：🚚 **明天到达**

您的订单已经在路上啦，预计明天就能送到您手中，请耐心等待哦！如果还有其他问题，比如需要修改地址或申请退款等，随时告诉我，我会尽力帮您解决～ 😊
用户：缓存服务有告警，帮我看看
路由：tech
回复：缓存服务当前存在告警，具体信息如下：

**告警详情：**
- **服务名称：** 缓存服务
- **告警内容：** 内存使用率 **85%**

**初步分析：**
内存使用率已达85%，接近警戒线（通常阈值为80%~90%），可能会导致缓存命中率下降或OOM风险。

**建议处理方案：**
1. **检查当前缓存容量和key数量**，确认是否有大量未过期或无效缓存堆积。
2. **评估是否需要扩容**或调整缓存淘汰策略（如LRU、TTL）。
3. 如需立即恢复，我可以尝试 **重启服务** 来释放部分内存（注意：重启会清空缓存数据，可能影响业务）。

需要我执行哪些操作？比如重启服务或进一步排查？
用户：我想退款ORD002，商品有质量问题
路由：cs
回复：我看到您的订单 **ORD002** 目前状态为 **"处理中"**，金额为 **1599元**，预计 **3天后发货**。由于订单尚未发货，我可以为您处理退款申请。

请问您方便具体描述一下商品的质量问题吗？这样我可以帮您提交退款申请。


### RAG Agent

In [ ]:
from langchain_core.tools import tool
from langchain_core.agents import create_agent
from langchain_core.messages import HumanMessage


# ===== 1. 模拟文档库 =====

documents = [
    {"id": 1, "title": "请假制度", "content": "员工每年享有15天年假。请提前3天在OA系统提交申请，直属领导审批后生效。病假需提供医院证明。"},
    {"id": 2, "title": "报销流程", "content": "差旅报销：出差结束后7日内提交，附发票和行程单。餐饮报销：每月汇总一次，限额500元/月。"},
    {"id": 3, "title": "考勤规定", "content": "上班时间9:00-18:00，弹性30分钟。迟到3次以内口头警告，超过3次扣绩效。"},
    {"id": 4, "title": "技术栈规范", "content": "后端使用Python/FastAPI，前端使用React/TypeScript。代码必须通过Code Review才能合并。"},
    {"id": 5, "title": "新人入职", "content": "入职第一天由HR引导办理工牌、开通账号。第二天由导师安排技术培训。试用期为3个月。"},
]

# ===== 2. 定义检索工具 =====

@tool
def search_documents(query: str) -> str:
    """搜索公司内部文档，输入搜索关键词。当用户问到公司制度、流程、规定时使用此工具。"""
    results = []
    query_lower = query.lower()
    for doc in documents:
        # 简单的关键词匹配
        if (query_lower in doc["title"].lower() or
            query_lower in doc["content"].lower() or
            any(c in doc["title"] or c in doc["content"] for c in query if '\u4e00' <= c <= '\u9fff')):
            results.append(f"【{doc['title']}】{doc['content']}")

    if not results:
        return f"未找到与'{query}'相关的文档。建议尝试其他关键词。"
    return "\n---\n".join(results)

@tool
def list_all_documents() -> str:
    """列出所有可用的文档标题列表。"""
    return "可用文档：" + "、".join([f"《{doc['title']}》" for doc in documents])


# ===== 3. 创建 RAG Agent =====

SYSTEM_PROMPT = """你是一个公司内部知识库助手。你的职责是回答员工关于公司制度、流程、规定的问题。

工作方式：
1. 先判断用户的问题是否需要查询文档
2. 如果需要，调用 search_documents 搜索相关文档
3. 根据搜索结果回答用户问题
4. 如果搜索结果不够，尝试换关键词重新搜索

不需要查询文档的情况：
- 简单问候（你好、谢谢等）
- 通用知识问题
- 你已经能确定答案的问题
"""

app = create_agent(
    ds_llm,
    [search_documents, list_all_documents],
    system_prompt=SYSTEM_PROMPT
)
# ===== 4. 测试 =====

questions = [
    "你好，你是做什么的？",
    "公司年假有多少天？怎么请假？",
    "出差报销流程是什么？",
    "公司用什么技术栈？",
    "帮我看看公司有哪些文档",
    "我想迟到10分钟，有问题吗？"
]

for question in questions:
    result = app.invoke({"messages": [HumanMessage(question)]})
    print(f"用户：{question}")
    print(f"AI：{result['messages'][-1].content}")
    print('='*30)

用户：你好，你是做什么的？
AI：你好！我是公司内部知识库助手，专门负责解答大家关于公司制度、流程、规定等方面的问题。

比如，如果你想知道：
- 请假的流程是怎么样的？
- 报销需要什么手续？
- 公司有哪些福利政策？
- 某个部门的联系方式是什么？

这些与公司内部相关的问题，我都可以帮你查找和解答！😊

请问有什么可以帮你的吗？
用户：公司年假有多少天？怎么请假？
AI：根据公司内部文档，为您解答如下：

---

### 📅 年假天数
- 每位员工**每年享有15天年假**。

### ✅ 请假流程
1. **提前申请**：请**提前3天**在 **OA系统** 提交年假申请。
2. **领导审批**：提交后由您的直属领导进行审批，审批通过后请假生效。

### 📌 注意事项
- 如果请**病假**，需要额外提供医院出具的证明。
- 如有其他疑问，欢迎随时问我！
用户：出差报销流程是什么？
AI：根据查询到的信息，以下是出差报销流程：

## 🧾 出差报销流程

**1. 提交时限**
- 出差结束后 **7日内** 提交报销申请

**2. 所需材料**
- 发票
- 行程单

**3. 其他报销相关规定**
- **餐饮报销**：每月汇总一次，限额 **500元/月**

---

如果您需要更详细的报销步骤（比如系统操作流程、审批环节等），欢迎告诉我，我可以换其他关键词进一步搜索。
用户：公司用什么技术栈？
AI：根据公司内部文档，公司的技术栈如下：

### **后端技术栈**
- **语言：** Python
- **框架：** FastAPI

### **前端技术栈**
- **语言：** TypeScript
- **框架：** React

此外，文档中还提到代码必须通过 **Code Review** 才能合并，说明团队有严格的代码审核流程。

如果你还想了解更具体的信息（比如数据库、部署工具等），我可以再帮你搜索一下~
用户：帮我看看公司有哪些文档
AI：目前公司知识库中有以下 **5 份文档**：

1. 📄 **《请假制度》** — 关于请假的相关规定
2. 📄 **《报销流程》** — 费用报销的流程说明
3. 📄 **《考勤规定》** — 考勤打卡等管理规定
4. 📄 **《技术栈规范》** — 公司技术栈相关规范
5. 📄 *